# Phase D — Knowledge Layer Lock & Data Dictionary

**Owner:** Member 2 (Mohammed)

Closes out the Member 2 phase. Two jobs:

1. **Validate** the knowledge layer (no nulls in critical columns, schema parity, reconciliation against Phase A).
2. **Document** every column in every processed file in a master `data_dictionary.md`.

Once validation passes, the knowledge layer is **locked** — Member 3 takes over and appends `cluster_label`, `risk_priority_score`, and any policy-sim columns they need.

## Outputs
- `reports/knowledge_layer_validation.csv` — pass/fail per check
- `data/processed/data_dictionary.md` — master column reference
- Individual `*_dict.csv` files per processed dataset

## 1. Imports & path setup

In [1]:
import sys
from datetime import datetime
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

from src.data_io import PROCESSED_DIR
from src import dictionary as dct

REPORTS = REPO_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print('Repo root:', REPO_ROOT)

Repo root: C:\Users\moham\Desktop\Term 3\Capstone\Repo\KPMG_Airbnb_Capstone


## 2. Load every processed file

In [2]:
bcn_clean = pd.read_csv(PROCESSED_DIR / 'barcelona' / 'barcelona_listings_clean.csv')
ldn_clean = pd.read_csv(PROCESSED_DIR / 'london' / 'london_listings_clean.csv')
bcn_feat = pd.read_csv(PROCESSED_DIR / 'barcelona' / 'barcelona_listings_features.csv')
ldn_feat = pd.read_csv(PROCESSED_DIR / 'london' / 'london_listings_features.csv')
bcn_monthly = pd.read_csv(PROCESSED_DIR / 'barcelona' / 'barcelona_monthly_metrics_clean.csv', parse_dates=['month_date'])
ldn_monthly = pd.read_csv(PROCESSED_DIR / 'london' / 'london_monthly_metrics_clean.csv', parse_dates=['month_date'])
kpis = pd.read_csv(PROCESSED_DIR / 'neighbourhood_kpis.csv')
ldn_borough = pd.read_csv(PROCESSED_DIR / 'london_borough_kpis.csv')
name_map = pd.read_csv(PROCESSED_DIR / 'subdivision_name_map.csv')

all_features = pd.concat([bcn_feat, ldn_feat], ignore_index=True)

shapes = {
    'barcelona_listings_clean': bcn_clean.shape,
    'london_listings_clean': ldn_clean.shape,
    'barcelona_listings_features': bcn_feat.shape,
    'london_listings_features': ldn_feat.shape,
    'barcelona_monthly_metrics': bcn_monthly.shape,
    'london_monthly_metrics': ldn_monthly.shape,
    'neighbourhood_kpis': kpis.shape,
    'london_borough_kpis': ldn_borough.shape,
    'subdivision_name_map': name_map.shape,
}
for k, v in shapes.items():
    print(f'{k:35s} {v}')

barcelona_listings_clean            (2594, 34)
london_listings_clean               (9643, 34)
barcelona_listings_features         (2594, 50)
london_listings_features            (9643, 50)
barcelona_monthly_metrics           (88821, 17)
london_monthly_metrics              (306822, 17)
neighbourhood_kpis                  (560, 27)
london_borough_kpis                 (32, 9)
subdivision_name_map                (18, 4)


## 3. Knowledge-layer validation

Gate Phase D. Any **FAIL** means we cannot hand off to Member 3.

In [3]:
checks = dct.validate_knowledge_layer(kpis, all_features)
checks.to_csv(REPORTS / 'knowledge_layer_validation.csv', index=False)
n_fail = (checks['status'] == 'FAIL').sum()
print(f'Checks run: {len(checks)}')
print(f'PASS: {(checks["status"]=="PASS").sum()}')
print(f'FAIL: {n_fail}')
assert n_fail == 0, 'Validation failed — see report.'
checks

Checks run: 19
PASS: 19
FAIL: 0


,check,status,detail
0,both cities present,PASS,"cities=['barcelona', 'london']"
1,no nulls in critical columns,PASS,nulls in: []
2,schema parity (BCN vs LDN),PASS,diff: set()
3,density reconciles (barcelona),PASS,"features=2354, kpis_sum=2354"
4,density reconciles (london),PASS,"features=9599, kpis_sum=9599"
5,entire_home reconciles (barcelona),PASS,"features=1414, kpis_sum=1414"
6,entire_home reconciles (london),PASS,"features=6504, kpis_sum=6504"
7,breach_90 reconciles (barcelona),PASS,"features=611, kpis_sum=611"
8,breach_90 reconciles (london),PASS,"features=1482, kpis_sum=1482"
9,"commercial_host_share in [0,1]",PASS,


## 4. Build per-file dictionary blocks

In [4]:
blocks = [
    dct.FileBlock(
        title='Barcelona / London — Listings (cleaned)',
        path='data/processed/{barcelona|london}/{city}_listings_clean.csv',
        unit_of_analysis='1 row = 1 listing',
        rows=len(bcn_clean) + len(ldn_clean),
        notes='Member 1 cleaning, plus Section 7.5 lat/lon and star_rating rescaling.',
        dictionary=dct.make_file_dictionary(bcn_clean, dct.LISTINGS_CLEAN, 'listings_clean'),
    ),
    dct.FileBlock(
        title='Barcelona / London — Monthly metrics',
        path='data/processed/{barcelona|london}/{city}_monthly_metrics_clean.csv',
        unit_of_analysis='1 row = 1 listing × 1 month',
        rows=len(bcn_monthly) + len(ldn_monthly),
        notes='Exploded from the months JSON column. March 2021 – February 2026.',
        dictionary=dct.make_file_dictionary(bcn_monthly, dct.MONTHLY_METRICS, 'monthly'),
    ),
    dct.FileBlock(
        title='Barcelona / London — Listings features (Phase A)',
        path='data/processed/{barcelona|london}/{city}_listings_features.csv',
        unit_of_analysis='1 row = 1 listing',
        rows=len(bcn_feat) + len(ldn_feat),
        notes='cleaned listings + 16 engineered features. Schema identical across both cities.',
        dictionary=dct.make_file_dictionary(all_features, dct.LISTINGS_FEATURES, 'features'),
    ),
    dct.FileBlock(
        title='Neighbourhood KPIs (Phase B — the knowledge layer)',
        path='data/processed/neighbourhood_kpis.csv',
        unit_of_analysis='1 row = 1 (city, geo_key)',
        rows=len(kpis),
        notes='Both cities stacked. The chatbot retrieves from this table.',
        dictionary=dct.make_file_dictionary(kpis, dct.NEIGHBOURHOOD_KPIS, 'kpis'),
    ),
    dct.FileBlock(
        title='London borough KPIs (Phase C)',
        path='data/processed/london_borough_kpis.csv',
        unit_of_analysis='1 row = 1 London borough',
        rows=len(ldn_borough),
        notes='Borough-level aggregation for LDN choropleths (geojson is borough-level).',
        dictionary=dct.make_file_dictionary(ldn_borough, dct.LONDON_BOROUGH_KPIS, 'borough'),
    ),
    dct.FileBlock(
        title='Subdivision name map (Phase C)',
        path='data/processed/subdivision_name_map.csv',
        unit_of_analysis='1 row = 1 rename rule',
        rows=len(name_map),
        notes='Apply with df.replace() before joining to geojson. BCN: 5 spelling fixes. LDN: 13 Westminster sub-areas.',
        dictionary=dct.make_file_dictionary(name_map, dct.SUBDIVISION_NAME_MAP, 'name_map'),
    ),
]
for b in blocks:
    print(f'{b.path:60s} {len(b.dictionary)} cols documented')

data/processed/{barcelona|london}/{city}_listings_clean.csv  34 cols documented
data/processed/{barcelona|london}/{city}_monthly_metrics_clean.csv 17 cols documented
data/processed/{barcelona|london}/{city}_listings_features.csv 50 cols documented
data/processed/neighbourhood_kpis.csv                        27 cols documented
data/processed/london_borough_kpis.csv                       9 cols documented
data/processed/subdivision_name_map.csv                      4 cols documented


## 5. Save per-file dictionary CSVs

Machine-readable column metadata for each processed file. Member 4's chatbot tools can load these to wire up automatic descriptions.

In [5]:
DICT_DIR = PROCESSED_DIR / 'dictionaries'
DICT_DIR.mkdir(parents=True, exist_ok=True)
naming = {
    'data/processed/{barcelona|london}/{city}_listings_clean.csv': 'listings_clean_dict.csv',
    'data/processed/{barcelona|london}/{city}_monthly_metrics_clean.csv': 'monthly_metrics_dict.csv',
    'data/processed/{barcelona|london}/{city}_listings_features.csv': 'listings_features_dict.csv',
    'data/processed/neighbourhood_kpis.csv': 'neighbourhood_kpis_dict.csv',
    'data/processed/london_borough_kpis.csv': 'london_borough_kpis_dict.csv',
    'data/processed/subdivision_name_map.csv': 'subdivision_name_map_dict.csv',
}
for b in blocks:
    out = DICT_DIR / naming[b.path]
    b.dictionary.to_csv(out, index=False)
    print(f'Saved {out.relative_to(REPO_ROOT)} ({len(b.dictionary)} columns)')

Saved data\processed\dictionaries\listings_clean_dict.csv (34 columns)
Saved data\processed\dictionaries\monthly_metrics_dict.csv (17 columns)
Saved data\processed\dictionaries\listings_features_dict.csv (50 columns)
Saved data\processed\dictionaries\neighbourhood_kpis_dict.csv (27 columns)
Saved data\processed\dictionaries\london_borough_kpis_dict.csv (9 columns)
Saved data\processed\dictionaries\subdivision_name_map_dict.csv (4 columns)


## 6. Assemble and save the master `data_dictionary.md`

In [6]:
today = datetime.utcnow().strftime('%Y-%m-%d')
md_text = dct.assemble_markdown(blocks, generated_on=today)
out_path = PROCESSED_DIR / 'data_dictionary.md'
out_path.write_text(md_text, encoding='utf-8')
print(f'Saved -> {out_path.relative_to(REPO_ROOT)}')
print(f'Size: {out_path.stat().st_size:,} bytes')

Saved -> data\processed\data_dictionary.md


Size: 54,847 bytes


## 7. Lock report — what Member 3 inherits

A short summary that goes alongside the table.

In [7]:
lock_report = (
    '# Knowledge Layer — Lock Report\n'
    f'_Locked on {today}_\n\n'
    '## Status\n\n'
    f'All {len(checks)} validation checks **PASS**.\n\n'
    '## Files Member 3 inherits\n\n'
    '| File | Rows | Cols | Purpose |\n'
    '|---|---|---|---|\n'
    f'| `data/processed/neighbourhood_kpis.csv` | {len(kpis):,} | {len(kpis.columns)} | Knowledge layer — cluster on this |\n'
    f'| `data/processed/barcelona/barcelona_listings_features.csv` | {len(bcn_feat):,} | {len(bcn_feat.columns)} | Per-listing features for the price model |\n'
    f'| `data/processed/london/london_listings_features.csv` | {len(ldn_feat):,} | {len(ldn_feat.columns)} | Same, for London |\n'
    f'| `data/processed/data_dictionary.md` | — | — | Column definitions for every file |\n'
    f'| `data/processed/dictionaries/` | — | — | Per-file CSV dictionaries (for chatbot tooling) |\n'
    f'| `data/processed/subdivision_name_map.csv` | {len(name_map):,} | — | Apply before joining geojson |\n\n'
    '## What Member 3 still needs to add\n\n'
    '- `cluster_label` per (city, geo_key) — saturated / emerging / low-impact\n'
    '- `risk_priority_score` — weighted 0–100 composite\n'
    '- Optionally widen the policy simulation columns at custom cap thresholds\n\n'
    '## Reproducibility\n\n'
    'Run notebooks 01 → 06 in order in the `kpmg-airbnb-capstone` env to regenerate everything.\n'
)
out_path = REPO_ROOT / 'reports' / 'knowledge_layer_lock_report.md'
out_path.write_text(lock_report, encoding='utf-8')
print(f'Saved -> {out_path.relative_to(REPO_ROOT)}')

Saved -> reports\knowledge_layer_lock_report.md


## 8. Final handover note

Phase D complete. Member 2 work locked.

- ✓ Knowledge layer validated
- ✓ Master `data_dictionary.md` generated
- ✓ Per-file dictionary CSVs saved in `data/processed/dictionaries/`
- ✓ Lock report at `reports/knowledge_layer_lock_report.md`

**Member 3 starts from `data/processed/neighbourhood_kpis.csv` and adds cluster + risk-score columns.**